# TAKTKRONE-I Evaluation Guide

Comprehensive benchmarking of trained models

## Benchmark Overview

TAKTKRONE-I includes 6 comprehensive benchmarks:

In [ ]:
from occlm.evaluation.benchmarks import (
    OCCTSummarization,
    DisruptionDiagnosis,
    RecoveryRanking,
    TopologyConsistency,
    SafetyGuard,
    RetrievalQA,
)

benchmarks = [
    ("OCC-SumEval", OCCTSummarization),
    ("DisruptionDiag", DisruptionDiagnosis),
    ("RecoveryRank", RecoveryRanking),
    ("TopoConsist", TopologyConsistency),
    ("SafetyGuard", SafetyGuard),
    ("RetrievalQA", RetrievalQA),
]

for name, benchmark_class in benchmarks:
    print(f"  {name:20s} - {benchmark_class.__doc__}")


## Load Model for Evaluation

In [ ]:
from occlm.serving.engine import AsyncOCCInferenceEngine
import asyncio

# Initialize engine
engine = AsyncOCCInferenceEngine(
    model_name="taktkrone-v0.1-sft-baseline",
    device="cuda"
)

# Load model
asyncio.run(engine.load())
print("Model loaded successfully!")


## Run Single Benchmark

Test on summarization:

In [ ]:
# Run summarization benchmark
benchmark = OCCTSummarization(model_engine=engine)

# Get test cases
test_cases = benchmark.get_test_cases()
print(f"Number of test cases: {len(test_cases)}")
print(f"\nExample test case:")
print(f"  Context: {test_cases[0]['context'][:100]}...")
print(f"  Reference: {test_cases[0]['reference_summary'][:100]}...")

# Run benchmark
print("\nRunning benchmark...")
results = asyncio.run(benchmark.run())

print(f"\nResults:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")


## Run All Benchmarks

Complete evaluation suite:

In [ ]:
import json
from datetime import datetime

# Run all benchmarks
all_results = {}

for name, BenchmarkClass in benchmarks:
    print(f"\nRunning {name}...")
    
    benchmark = BenchmarkClass(model_engine=engine)
    results = asyncio.run(benchmark.run())
    all_results[name] = results
    
    # Print summary
    print(f"  Results: {results}")

# Save results
report = {
    "timestamp": datetime.now().isoformat(),
    "model": "taktkrone-v0.1-sft-baseline",
    "benchmarks": all_results
}

with open("benchmark_results.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nResults saved to benchmark_results.json")


## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("benchmark_results.json") as f:
    report = json.load(f)

# Plot scores for each benchmark
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, results) in enumerate(report["benchmarks"].items()):
    ax = axes[idx]
    
    metrics = list(results.keys())
    values = list(results.values())
    
    ax.bar(metrics, values, color='steelblue')
    ax.set_title(name)
    ax.set_ylabel('Score (0-1)')
    ax.set_ylim([0, 1])
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("benchmark_results.png")
plt.show()

print("Visualization saved to benchmark_results.png")


## Compare Multiple Models

Benchmark different model versions:

In [ ]:
# Compare two models
model_configs = [
    ("SFT Baseline", "taktkrone-v0.1-sft-baseline"),
    ("LoRA Fine-tuned", "taktkrone-v0.1-lora"),
]

comparison_results = {}

for model_name, model_path in model_configs:
    print(f"\nEvaluating {model_name}...")
    
    # Load model
    engine = AsyncOCCInferenceEngine(model_name=model_path)
    asyncio.run(engine.load())
    
    # Run benchmarks
    model_results = {}
    for name, BenchmarkClass in benchmarks:
        benchmark = BenchmarkClass(model_engine=engine)
        results = asyncio.run(benchmark.run())
        # Get average score
        model_results[name] = np.mean(list(results.values()))
    
    comparison_results[model_name] = model_results

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

benchmark_names = list(comparison_results[list(comparison_results.keys())[0]].keys())
x = np.arange(len(benchmark_names))
width = 0.35

for i, (model_name, scores) in enumerate(comparison_results.items()):
    values = [scores[b] for b in benchmark_names]
    ax.bar(x + i*width, values, width, label=model_name)

ax.set_ylabel('Average Score')
ax.set_title('Model Comparison Across Benchmarks')
ax.set_xticks(x + width/2)
ax.set_xticklabels(benchmark_names, rotation=45)
ax.legend()
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig("model_comparison.png")
plt.show()

print("Comparison saved to model_comparison.png")


## Detailed Analysis

Deep dive into specific benchmarks:

In [ ]:
# Detailed analysis of classification benchmark
benchmark = DisruptionDiagnosis(model_engine=engine)
results = asyncio.run(benchmark.run())

print("Classification Results:")
print(f"  Overall Accuracy: {results['accuracy']:.4f}")
print(f"  Per-class F1 scores:")
for disruption_type, f1 in results.get('per_class_f1', {}).items():
    print(f"    {disruption_type}: {f1:.4f}")

# Test cases where model failed
print(f"\nMisclassified examples:")
test_cases = benchmark.get_test_cases()
for i, test_case in enumerate(test_cases[:3]):
    print(f"  {i+1}. {test_case['incident'][:60]}...")
    print(f"     Expected: {test_case['disruption_type']}")
